# UNSW Battlecode self-play PPO training

Trains a shared-weight RL policy for the UNSW Battlecode 2026 ("Dragons") game
by self-play PPO on a from-scratch simulator, then exports it to a
plain-NumPy bot you can run with `unswbc` or submit to the ladder.

Rules: https://game.battlecode.au/docs/overview

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

In [ ]:
!nvidia-smi

## 1. Get the code

Replace `REPO_URL` below if you forked this.

In [ ]:
REPO_URL = "https://github.com/Murat65536/unswbc-rl.git"
!git clone "$REPO_URL" repo
%cd repo
!pip install -q -r requirements.txt pytest unswbc

## 2. Sanity check

Runs the engine unit tests and the training/bot observation-encoding
contract test before burning any GPU time.

In [ ]:
!unswbc init python bots/rl_bot  # regenerate the real helper.py the contract test needs
!python -m pytest tests -q

## 3. Persist checkpoints to Drive

Colab runtimes get reclaimed; this makes training resumable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CHECKPOINT_DIR = "/content/drive/MyDrive/unswbc-rl-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 4. Quick smoke run

Small and fast (~1-2 min), just to see the training loop move before
committing to a long run. Expect episode length and win rates to be
noisy/near-random this early -- that's fine.

In [ ]:
!python -m rl.train \
  --num-envs 32 --team-size 1 --width 14 --height 14 --max-rounds 150 --kelp-prob 0.0 \
  --iterations 30 --rollout-length 64 --minibatch-size 2048 \
  --checkpoint-dir /tmp/smoke_ckpt --device cuda

## 5. Real training run

A rough curriculum: start small/easy (no kelp, 1 dragon/team) so the policy
learns to chase pearls and not suicide, then scale up map size, team size
and hazards, resuming from the previous stage's checkpoint each time. Tune
iteration counts to your time budget -- this is a starting point, not a
tuned recipe.

In [ ]:
# Stage 1: learn the basics.
!python -m rl.train \
  --num-envs 48 --team-size 1 --width 16 --height 16 --max-rounds 250 --kelp-prob 0.03 \
  --iterations 800 --rollout-length 128 --minibatch-size 4096 \
  --checkpoint-dir "$CHECKPOINT_DIR/stage1" --checkpoint-every 25 \
  --log-csv "$CHECKPOINT_DIR/stage1/log.csv" --device cuda

In [ ]:
# Stage 2: bigger map, multiple dragons per team, resuming from stage 1.
!python -m rl.train \
  --num-envs 48 --team-size 3 --width 24 --height 24 --max-rounds 400 --kelp-prob 0.05 \
  --iterations 2000 --rollout-length 128 --minibatch-size 4096 \
  --resume "$CHECKPOINT_DIR/stage1/latest.pt" \
  --checkpoint-dir "$CHECKPOINT_DIR/stage2" --checkpoint-every 25 \
  --log-csv "$CHECKPOINT_DIR/stage2/log.csv" --device cuda

If the runtime disconnects mid-stage, rerun the same cell with
`--resume "$CHECKPOINT_DIR/stageN/latest.pt"` pointed at its own stage dir
to pick back up.

## 6. Plot progress

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{CHECKPOINT_DIR}/stage2/log.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.plot(x="iteration", y="mean_final_length", ax=axes[0], title="mean final dragon length")
df.plot(x="iteration", y=["win_rate_a", "win_rate_b", "draw_rate"], ax=axes[1], title="outcomes")
df.plot(x="iteration", y="mean_episode_reward", ax=axes[2], title="mean reward")
plt.tight_layout()

## 7. Export to a submittable bot

Produces a plain-NumPy `unswbc` bot project -- no PyTorch, since the judge's
sandbox doesn't have it.

In [ ]:
!python -m export.export_bot --checkpoint "$CHECKPOINT_DIR/stage2/latest.pt" --out bots/rl_bot_trained
!unswbc run bots/maps/arena.map bots/rl_bot_trained bots/rl_bot_trained --sandbox -v

## 8. Submit (optional)

Needs an API key from your team page (game.battlecode.au/team).

In [ ]:
# !unswbc auth set bc_...
# !unswbc submit bots/rl_bot_trained -n rl-v1 -d "PPO self-play, stage 2"